# 06 - Query Refinement Service Test

هذا الـ Notebook يختبر Query Refinement كخدمة مستقلة.

أمثلة الخدمات:
- Query Processing
- PRF إذا كانت نتائج أولية متوفرة
- Word2Vec Expansion إذا كان نموذج Word2Vec موجوداً

In [ ]:
from pathlib import Path
import sys, os, json, sqlite3, subprocess, time
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# يستطيع صديقك تغيير هذا المتغير إذا كانت artifacts خارج المشروع.
# مثال في PowerShell قبل فتح notebook:
# $env:IR_ARTIFACT_ROOT="E:\\ir_project_artifacts"
ARTIFACT_ROOT = Path(os.environ.get("IR_ARTIFACT_ROOT", r"E:\ir_project_artifacts"))


def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return Path(paths[0])

DB_PATH = first_existing([
    PROJECT_ROOT / "data" / "documents.sqlite",
    PROJECT_ROOT / "data" / "documents.db",
    ARTIFACT_ROOT / "documents.sqlite",
    ARTIFACT_ROOT / "documents.db",
])

TERRIER_INDEX_PATH = first_existing([
    PROJECT_ROOT / "indexes" / "terrier_medline",
    PROJECT_ROOT / "data" / "indexes" / "terrier_medline",
    ARTIFACT_ROOT / "indexes" / "terrier_medline",
])

BERT_INDEX_DIR = first_existing([
    PROJECT_ROOT / "indexes" / "faiss_bert",
    PROJECT_ROOT / "indexes" / "faiss_bert_full",
    PROJECT_ROOT / "data" / "rag_artifacts",
    ARTIFACT_ROOT / "indexes" / "faiss_bert_full",
    ARTIFACT_ROOT / "indexes" / "faiss_bert",
])

WORD2VEC_INDEX_DIR = first_existing([
    PROJECT_ROOT / "indexes" / "faiss_word2vec",
    PROJECT_ROOT / "indexes" / "faiss_word2vec_full",
    ARTIFACT_ROOT / "indexes" / "faiss_word2vec_full",
    ARTIFACT_ROOT / "indexes" / "faiss_word2vec",
])

REPORTS_DIR = PROJECT_ROOT / "reports" / "evaluation_notebook"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT       =", PROJECT_ROOT)
print("ARTIFACT_ROOT      =", ARTIFACT_ROOT)
print("DB_PATH            =", DB_PATH, "exists=", DB_PATH.exists())
print("TERRIER_INDEX_PATH =", TERRIER_INDEX_PATH, "exists=", TERRIER_INDEX_PATH.exists())
print("BERT_INDEX_DIR     =", BERT_INDEX_DIR, "exists=", BERT_INDEX_DIR.exists())
print("WORD2VEC_INDEX_DIR =", WORD2VEC_INDEX_DIR, "exists=", WORD2VEC_INDEX_DIR.exists())

In [ ]:
from src.query_refinement import QueryRefinementConfig, QueryRefinementService

query = "breast cancer gene mutation"
print("Original query:", query)

In [ ]:
# اختبار بسيط بدون Word2Vec، حسب الطرق المدعومة في المشروع.
# إذا تغيرت أسماء methods في المشروع، اعرضها من models.py أو عدل method.
methods_to_try = ["prf", "word2vec", "context_word2vec"]

for method in methods_to_try:
    print("="*90)
    print("Trying refinement method:", method)
    try:
        config = QueryRefinementConfig(method=method)
        kwargs = {"config": config}
        if "word2vec" in method:
            kwargs["word2vec_index_dir"] = str(WORD2VEC_INDEX_DIR)
        service = QueryRefinementService(**kwargs)
        result = service.refine(query=query, feedback_documents=[])
        print("processed_query:", result.processed_query)
        print("refined_query  :", result.refined_query)
        print("original_terms :", result.original_terms)
        print("word2vec_terms :", result.word2vec_terms)
        print("prf_terms      :", result.prf_terms)
    except Exception as exc:
        print("FAILED:", type(exc).__name__, exc)